In [1]:
import sys
sys.path.append('..')

import pandas as pd

from utils.db_utils import write_table, read_table

ModuleNotFoundError: No module named 'utils'

In [ ]:
GDP_URL_DATA = 'https://storage.dosm.gov.my/gdp/gdp_state_real_supply.parquet'
gdp_df = pd.read_parquet(GDP_URL_DATA)

LOOKUP_URL_DATA = 'https://storage.dosm.gov.my/gdp/gdp_lookup.parquet'
lookup_df = pd.read_parquet(LOOKUP_URL_DATA)

In [ ]:
for col in ["series", "state", "sector"]:
    unique_vals = sorted(gdp_df[col].dropna().unique())
    print(f"\n{col} ({len(unique_vals)} unique values)")
    print(unique_vals)

In [ ]:
gdp_df.info()

In [ ]:
def preprocess_gdp(gdp_df, lookup_df):

    # -------------------------------------------------
    # 1. Clean original df
    # -------------------------------------------------

    gdp_df["state"] = gdp_df["state"].replace({
        "Supra": "W.P. Putrajaya"
    })
    
    gdp_df = gdp_df[gdp_df["date"].dt.year >= 2016].copy()

    # -------------------------------------------------
    # 2. Pivot 'series' column into abs and growth_yoy
    # -------------------------------------------------
    df = (
        gdp_df
        .pivot_table(
            index=["state", "date", "sector"],
            columns="series",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    df.columns.name = None

    df = df.rename(columns={
        "sector": "code",
        "abs": "RM(million)"
    })
    
    # -------------------------------------------------
    # 3. Join lookup table to get sector description
    # -------------------------------------------------
    df = df.merge(
        lookup_df[["code", "desc_en"]],
        on="code",
        how="left"
    )

    df = df.rename(columns={"desc_en": "sector"})
    
    
    df["sector"] = df["sector"].replace({
        "Mining and quarrying": "Mining and Quarrying"
    })

    df = df[~df["sector"].isin(["GDP at purchasers' prices", "(plus) Import duties"])].copy()
    
    df_final = df[
        ["state", "date", "sector", "RM(million)", "growth_yoy"]
    ]
    df_final["growth_yoy"] = df_final["growth_yoy"].fillna(0).astype(float)
    return df_final

In [ ]:
df_final = preprocess_gdp(gdp_df, lookup_df)
df_final.head(20)

In [ ]:
# -------------------------------
# 1. Prepare the 2024 data (Get it from gdp 2024, datagov don't have 2024)
# -------------------------------
states = [
    "Johor","Kedah","Kelantan","Melaka","Negeri Sembilan",
    "Pahang","Pulau Pinang","Perak","Perlis","Selangor",
    "Terengganu","Sabah","Sarawak","W.P. Kuala Lumpur","W.P. Labuan", "W.P. Putrajaya"
]

sectors = ["Agriculture","Mining and Quarrying","Manufacturing","Construction","Services"]

# RM(million) values for 2024
rm_values = [
    [17934, 771, 46000, 6374, 85161],
    [5728, 118, 16135, 1422, 30310],
    [5764, 463, 1293, 562, 20534],
    [4656, 70, 17588, 1352, 25171],
    [3438, 214, 20399, 1492, 28656],
    [15312, 604, 14531, 2564, 35536],
    [2197, 162, 56063, 3865, 58433],
    [12094, 480, 16769, 2248, 54579],
    [1121, 34, 491, 181, 4608],
    [4880, 931, 125784, 22502, 263947],
    [2941, 236, 14628, 1370, 20710],
    [12205, 18551, 6103, 2958, 44209],
    [15059, 31098, 38190, 6082, 57264],
    [0, 158, 6689, 13454, 244152], # Kuala Lumpur, a -> 0
    [129, 0, 1373, 156, 6840],      # Labuan, - -> 0
    [0, 44541, 0, 0, 0]                 # Putrajaya, - -> 0
]

# growth_yoy values for 2024
growth_values = [
    [4.2, 11.3, 4.2, 42.7, 6.0],
    [4.0, -3.0, 6.6, -11.1, 3.8],
    [2.8, 8.2, 2.2, 20.7, 3.4],
    [-1.0, 6.6, 3.8, 30.6, 4.8],
    [9.8, -5.2, 3.9, 9.1, 4.3],
    [8.4, -3.9, 3.5, 12.9, 4.9],
    [0.1, 5.5, 4.0, 14.4, 5.0],
    [3.6, -2.9, 5.1, 11.5, 4.1],
    [2.9, 2.7, 1.4, 16.9, 3.3],
    [6.6, 11.8, 5.1, 13.2, 6.3],
    [3.9, 13.6, 3.9, 17.5, 4.0],
    [-3.4, -5.0, 1.2, 18.8, 4.2],
    [0.5, 4.1, 1.3, 18.7, 4.9],
    [0, 2.9, 4.3, 21.1, 5.5],  # Kuala Lumpur, .. -> 0
    [2.3, 0, -0.7, 5.3, 6.4],  # Labuan, - -> 0
    [0, 1.0, 0, 0, 0]                 # Putrajaya, - -> 0
]

# -------------------------------
# 2. Construct a list of dicts
# -------------------------------

data_list = []

for i, state in enumerate(states):
    for j, sector in enumerate(sectors):
        data_list.append({
            "state": state,
            "date": pd.to_datetime("2024-01-01"),
            "sector": sector,
            "RM(million)": rm_values[i][j],
            "growth_yoy": growth_values[i][j]
        })


df_2024 = pd.DataFrame(data_list)

# -------------------------------
# 3. Append to your existing df_final
# -------------------------------
df_2024["date"] = df_2024["date"].astype(df_final["date"].dtype)
df_2024["RM(million)"] = df_2024["RM(million)"].astype(df_final["RM(million)"].dtype)
df_final = pd.concat([df_final, df_2024], ignore_index=True)
df_final = df_final.sort_values(by=["state", "date"]).reset_index(drop=True)

In [ ]:
write_table(df_final, "sc_bronze", "datagov_gdp")